# Three-dimensional geodesic deposition

This compact spherical workflow exactly deposits a tetrahedral sheet source onto a layered geodesic target mesh. Increase the beam, ray, and angular counts in the configuration for production studies.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pyGATH.fields import (
    build_geodesic_deposition_mesh_from_grid,
    deposit_simplicial_power_to_mesh,
    simplicialise_sheet_fields,
)
from pyGATH.io import load_simulation_config

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
simulation = load_simulation_config(
    root / "configs/example_configs/three_dimensional_geodesic_deposition.toml"
)

In [ ]:
with simulation.reporting():
    grid = simulation.build_grid()
    beams = simulation.load_beams()
    initial_rays = simulation.initialize_rays(grid, beams=beams)
    trace = simulation.trace_rays(initial_rays, grid)
    source = simplicialise_sheet_fields(
        trace.sheet_fields, dimension=3, fields="inverse_brems_deposition"
    )
    target = build_geodesic_deposition_mesh_from_grid(grid, maximum_angular_cells=40)
    deposition = deposit_simplicial_power_to_mesh(
        source, target, source_batch_size=1024
    )
print(f"{source.mesh.nsimplices:,} source tetrahedra per sheet")
print(f"{target.ncells:,} target tetrahedra")
print(f"conservation error={deposition.conservation_error:.3e} W")

In [ ]:
layer_power = np.bincount(
    target.simplex_radial_layer,
    weights=np.asarray(deposition.cell_power),
    minlength=target.nradial,
)
radii_um = 0.5 * (target.radial_boundaries[:-1] + target.radial_boundaries[1:]) * 1e6
plt.plot(radii_um, layer_power, marker="o")
plt.xlabel("radius [um]")
plt.ylabel("deposited power per radial layer [W]")
plt.grid(alpha=0.25)